# Does Retrieval Help? Grounding a Small LLM in Trusted Healthcare Guidance

**Research question:** Does giving an LLM relevant passages from trusted healthcare guidance improve the **accuracy** and **evidence support** of its answers?

This notebook is a complete, beginner-friendly experiment. You will build a small **RAG (Retrieval-Augmented Generation)** system and compare it against the same model answering from memory alone.

### Experimental design

| Condition | What the model sees | Why it's here |
|---|---|---|
| **A. Closed-book** | The question only | Baseline: what the model "knows" |
| **B. RAG** | Question + top-3 retrieved guidance passages | The intervention we're testing |
| **C. RAG (gold removed)** | Question + top-3 passages, *excluding* the document the reference answer came from | Robustness check: does RAG still help when the exact answer isn't in the library? |

Same model, same questions, same decoding settings — only the context changes. That makes this a **paired, controlled comparison**.

### Hypotheses
- **H1 (accuracy):** RAG answers are closer to the trusted reference answer than closed-book answers.
- **H2 (evidence support):** A higher share of RAG answer sentences are *entailed* by trusted guidance, and fewer contradict it.

### What you need
- A Google account. In Colab: **Runtime → Change runtime type → T4 GPU** (free tier is enough).
- ~30–40 minutes of runtime for 100 questions.

> ⚠️ This is a research exercise about model behaviour. Nothing generated here is medical advice.

## 0. How the pipeline works

```
 MedQuAD answers (NIH sources) ──► chunk ──► embed ──► FAISS index
                                                          │
 question ─────────────────────────────► retrieve top-k ──┘
     │                                         │
     ├──► LLM (no context)        ──► answer A │
     └──► LLM (question + passages) ──► answer B / C
                                               │
 evaluate vs. reference answer: ROUGE-L, semantic similarity,
 NLI-based evidence support & contradiction, citations → statistics
```

## 1. Setup
Install the few libraries Colab doesn't already have, then confirm the GPU is on.

In [ ]:
!pip -q install sentence-transformers faiss-cpu rouge-score

In [ ]:
import os, re, json, random, textwrap, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Switch to a GPU: Runtime → Change runtime type → T4 GPU, then re-run.")

## 2. Configuration
Every experimental choice lives here, so it is easy to report in your write-up and easy to change later.
Start with `N_QUESTIONS = 20` for a quick test run, then raise it to 100+ for the real experiment.

In [ ]:
SEED          = 42
N_QUESTIONS   = 100          # use 20 for a first test run
MODEL_ID      = "Qwen/Qwen2.5-1.5B-Instruct"   # try "Qwen/Qwen2.5-3B-Instruct" later
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"       # retriever
EVAL_EMBED    = "sentence-transformers/all-MiniLM-L6-v2"  # different model for evaluation (avoids bias)
NLI_MODEL     = "cross-encoder/nli-deberta-v3-base"
TOP_K         = 3            # passages given to the model
CHUNK_WORDS   = 120
CHUNK_OVERLAP = 30
MAX_NEW_TOKENS = 200
BATCH_SIZE    = 8

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Load the trusted knowledge source
**MedQuAD** contains question–answer pairs collected from 12 NIH websites (e.g., MedlinePlus, NCI, GARD). Each answer acts as one **"guidance document"**.

We use it two ways:
1. All answers form the **retrieval library**.
2. A sample of questions forms the **test set**; each question's own answer is the **reference** (ground truth).

In [ ]:
from datasets import load_dataset

ds = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
df = ds.to_pandas()
df.columns = [c.lower().strip() for c in df.columns]   # -> qtype, question, answer
print(df.columns.tolist(), len(df))

df = df.dropna(subset=["question", "answer"])
df["question"] = df["question"].str.strip()
df["answer"]   = df["answer"].str.strip()
df = df.drop_duplicates(subset=["answer"]).reset_index(drop=True)
df["doc_id"]  = df.index
df["n_words"] = df["answer"].str.split().str.len()
print("Unique guidance documents:", len(df))
df.head(3)

### Choose the test questions
We keep questions whose reference answer is a reasonable length (40–300 words) so the comparison is fair, and drop questions that appear more than once.

In [ ]:
cands = df[(df.n_words.between(40, 300))]
cands = cands[~cands.duplicated(subset=["question"], keep=False)]
eval_df = cands.sample(n=N_QUESTIONS, random_state=SEED).reset_index(drop=True)
eval_df["qid"] = range(len(eval_df))
print(eval_df["qtype"].value_counts().head(10))
eval_df[["qid", "qtype", "question"]].head()

## 4. Build the retriever
Long documents are split into overlapping **chunks** (~120 words). Each chunk is turned into an **embedding** (a vector capturing meaning), and stored in a **FAISS** index for fast similarity search.

*Key idea:* a question and a passage that answers it end up with similar vectors.

In [ ]:
def chunk_text(text, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    words = text.split()
    if len(words) <= size:
        return [" ".join(words)]
    chunks, step = [], size - overlap
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + size]))
        if start + size >= len(words):
            break
    return chunks

rows = []
for r in df.itertuples():
    for j, ch in enumerate(chunk_text(r.answer)):
        rows.append({"chunk_id": len(rows), "doc_id": r.doc_id, "text": ch})
chunks_df = pd.DataFrame(rows)
print(f"{len(df)} documents -> {len(chunks_df)} chunks")

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
t0 = time.time()
chunk_emb = embedder.encode(chunks_df["text"].tolist(), batch_size=256,
                            normalize_embeddings=True, show_progress_bar=True)
chunk_emb = np.asarray(chunk_emb, dtype="float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])   # inner product = cosine (vectors are normalised)
index.add(chunk_emb)
print(f"Indexed {index.ntotal} chunks in {time.time()-t0:.0f}s")

In [ ]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # recommended for bge models

def retrieve(question, k=TOP_K, exclude_doc=None):
    q = embedder.encode([QUERY_PREFIX + question], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k * 20)
    out = []
    for s, i in zip(scores[0], idx[0]):
        if i < 0:
            continue
        row = chunks_df.iloc[i]
        if exclude_doc is not None and row.doc_id == exclude_doc:
            continue
        out.append({"text": row.text, "doc_id": int(row.doc_id), "score": float(s)})
        if len(out) == k:
            break
    return out

# Try it
demo = eval_df.iloc[0]
print("Q:", demo.question)
for i, p in enumerate(retrieve(demo.question), 1):
    print(f"\n[{i}] (score {p['score']:.2f}, doc {p['doc_id']}) {p['text'][:200]}...")

### Check the retriever before trusting it
**Recall@k** = how often the reference document appears in the top-k results. If retrieval is poor, RAG can't help — so we measure it separately. (Report this number!)

In [ ]:
eval_df["passages_rag"]  = [retrieve(q) for q in tqdm(eval_df.question)]
eval_df["passages_hard"] = [retrieve(q, exclude_doc=d) for q, d in tqdm(zip(eval_df.question, eval_df.doc_id), total=len(eval_df))]

hit = [d in [p["doc_id"] for p in ps] for d, ps in zip(eval_df.doc_id, eval_df.passages_rag)]
print(f"Retrieval Recall@{TOP_K}: {np.mean(hit):.2%}")

## 5. Load the LLM
We use a small open instruction-tuned model that fits on a free T4 GPU and needs no access token.
**Greedy decoding** (`do_sample=False`) makes outputs deterministic, so differences come from the context, not randomness.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"               # needed for batched generation
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Loaded", MODEL_ID)

### Prompts
The two prompts are deliberately as similar as possible. The RAG prompt adds: use only the passages, cite them as `[1]`, and say so if the passages are insufficient (this lets us measure **abstention**).

In [ ]:
CLOSED_SYS = ("You are a careful health information assistant. Answer the question accurately "
              "in 3-5 sentences of plain language.")
RAG_SYS = ("You are a careful health information assistant. Answer the question accurately "
           "in 3-5 sentences of plain language, using ONLY the numbered passages provided. "
           "Cite the passage(s) you use like [1] or [2]. If the passages do not contain the "
           "answer, say \"The provided passages do not contain this information.\"")

def build_messages(question, passages=None):
    if passages is None:
        return [{"role": "system", "content": CLOSED_SYS},
                {"role": "user", "content": f"Question: {question}"}]
    ctx = "\n\n".join(f"[{i+1}] {p['text']}" for i, p in enumerate(passages))
    return [{"role": "system", "content": RAG_SYS},
            {"role": "user", "content": f"Passages:\n{ctx}\n\nQuestion: {question}"}]

@torch.no_grad()
def generate_batch(message_lists):
    prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in message_lists]
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         temperature=None, top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
    new_tokens = out[:, enc["input_ids"].shape[1]:]
    return [t.strip() for t in tok.batch_decode(new_tokens, skip_special_tokens=True)]

print(generate_batch([build_messages(demo.question)])[0])

## 6. Run the experiment
All three conditions for every question. Results are saved to CSV so you don't lose them if Colab disconnects.

In [ ]:
CONDITIONS = {
    "closed_book": lambda r: build_messages(r.question),
    "rag":         lambda r: build_messages(r.question, r.passages_rag),
    "rag_hard":    lambda r: build_messages(r.question, r.passages_hard),
}

records = []
t0 = time.time()
for cond, make_msgs in CONDITIONS.items():
    rows_ = list(eval_df.itertuples())
    for i in tqdm(range(0, len(rows_), BATCH_SIZE), desc=cond):
        batch = rows_[i:i + BATCH_SIZE]
        answers = generate_batch([make_msgs(r) for r in batch])
        for r, a in zip(batch, answers):
            psg = None if cond == "closed_book" else (r.passages_rag if cond == "rag" else r.passages_hard)
            records.append({"qid": r.qid, "qtype": r.qtype, "question": r.question,
                            "reference": r.answer, "gold_doc": r.doc_id, "condition": cond,
                            "passages": json.dumps(psg), "answer": a})
res = pd.DataFrame(records)
res.to_csv("generations.csv", index=False)
print(f"Done: {len(res)} answers in {(time.time()-t0)/60:.1f} min")

In [ ]:
# Read a few side by side - always look at your data!
q = res.qid.iloc[3]
for _, r in res[res.qid == q].iterrows():
    print(f"=== {r.condition} ===\n{textwrap.fill(r.answer, 110)}\n")
print("=== REFERENCE ===\n" + textwrap.fill(res[res.qid == q].reference.iloc[0][:600], 110))

In [ ]:
# Free GPU memory for the evaluation models
del model; torch.cuda.empty_cache()

## 7. Evaluate
We need numbers for both parts of the research question.

**Accuracy (H1)** — how close is the answer to the trusted reference?
- **ROUGE-L F1:** word-sequence overlap. Simple, widely reported, but rewards copying.
- **Semantic similarity:** cosine similarity of embeddings — captures same meaning in different words. Uses a *different* embedding model from the retriever to avoid favouring RAG.

**Evidence support (H2)** — is each claim backed by trusted guidance?
We split each answer into sentences and use an **NLI (Natural Language Inference)** model, which judges whether a passage *entails*, *contradicts*, or is *neutral* to a sentence.
- **Support rate:** % of answer sentences entailed by the trusted reference document.
- **Contradiction rate:** % of sentences that contradict it (a safety-relevant error).
- **Faithfulness** (RAG only): % of sentences entailed by the passages the model was given.

**Behaviour:** citation rate and abstention rate.

In [ ]:
from rouge_score import rouge_scorer
from sentence_transformers import CrossEncoder

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
eval_embedder = SentenceTransformer(EVAL_EMBED, device=DEVICE)
nli = CrossEncoder(NLI_MODEL, device=DEVICE)
# Label order for this model (see its model card): 0=contradiction, 1=entailment, 2=neutral
CON, ENT = 0, 1

ABSTAIN_PAT = re.compile(r"do(es)? not contain|not (provided|mentioned|available) in the passages|cannot answer", re.I)
CITE_PAT = re.compile(r"\[(\d+)\]")

def split_sentences(text):
    text = CITE_PAT.sub("", text)
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s.strip() for s in sents if len(s.split()) >= 4]

def nli_scores(answer, premises, thr=0.5):
    sents = split_sentences(answer)
    if not sents or not premises:
        return np.nan, np.nan
    pairs = [(p, s) for s in sents for p in premises]
    probs = np.array(nli.predict(pairs, apply_softmax=True, batch_size=32, show_progress_bar=False))
    probs = probs.reshape(len(sents), len(premises), 3)
    ent = probs[:, :, ENT].max(axis=1)
    con = probs[:, :, CON].max(axis=1)
    return float((ent > thr).mean()), float(((con > thr) & (ent <= thr)).mean())

gold_chunks = chunks_df.groupby("doc_id")["text"].apply(list).to_dict()

In [ ]:
ans_emb = eval_embedder.encode(res.answer.tolist(), normalize_embeddings=True, batch_size=64)
ref_emb = eval_embedder.encode(res.reference.tolist(), normalize_embeddings=True, batch_size=64)
res["semantic_sim"] = (ans_emb * ref_emb).sum(axis=1)
res["rougeL"] = [scorer.score(ref, ans)["rougeL"].fmeasure for ref, ans in zip(res.reference, res.answer)]
res["abstained"] = res.answer.str.contains(ABSTAIN_PAT)
res["cited"] = [bool(CITE_PAT.search(a)) if c != "closed_book" else np.nan for a, c in zip(res.answer, res.condition)]

support, contra, faith = [], [], []
for r in tqdm(res.itertuples(), total=len(res), desc="NLI"):
    s, c = nli_scores(r.answer, gold_chunks[r.gold_doc])
    support.append(s); contra.append(c)
    psg = json.loads(r.passages)
    faith.append(nli_scores(r.answer, [p["text"] for p in psg])[0] if psg else np.nan)
res["support_rate"], res["contradiction_rate"], res["faithfulness"] = support, contra, faith
res.to_csv("results_scored.csv", index=False)

METRICS = ["semantic_sim", "rougeL", "support_rate", "contradiction_rate"]
summary = res.groupby("condition")[METRICS + ["faithfulness", "abstained", "cited"]].mean().round(3)
summary.loc[["closed_book", "rag", "rag_hard"]]

## 8. Statistics: is the difference real?
Because every question is answered under every condition, we use **paired** tests:
- **Mean difference with a 95% bootstrap confidence interval** — if the interval excludes 0, the effect is reliable.
- **Wilcoxon signed-rank test** — a non-parametric paired test (no normality assumption).
- **Holm correction**, because we run several tests at once.

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

def paired_compare(metric, a, b, n_boot=5000):
    piv = res.pivot(index="qid", columns="condition", values=metric)[[a, b]].dropna()
    diff = (piv[b] - piv[a]).values
    rng = np.random.default_rng(SEED)
    boots = rng.choice(diff, size=(n_boot, len(diff)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    try:
        p = wilcoxon(piv[b], piv[a]).pvalue
    except ValueError:      # all differences zero
        p = 1.0
    return {"metric": metric, "comparison": f"{b} vs {a}", "n": len(diff),
            "mean_diff": diff.mean(), "ci_low": lo, "ci_high": hi, "p_raw": p}

tests = pd.DataFrame([paired_compare(m, "closed_book", b) for b in ["rag", "rag_hard"] for m in METRICS])
tests["p_holm"] = multipletests(tests.p_raw, method="holm")[1]
tests["significant"] = tests.p_holm < 0.05
tests.round(4)

In [ ]:
order = ["closed_book", "rag", "rag_hard"]
labels = {"semantic_sim": "Semantic similarity", "rougeL": "ROUGE-L F1",
          "support_rate": "Evidence support rate", "contradiction_rate": "Contradiction rate"}
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
rng = np.random.default_rng(SEED)
for ax, m in zip(axes, METRICS):
    means, errs = [], []
    for c in order:
        v = res.loc[res.condition == c, m].dropna().values
        b = rng.choice(v, size=(2000, len(v))).mean(axis=1)
        means.append(v.mean()); errs.append([v.mean() - np.percentile(b, 2.5), np.percentile(b, 97.5) - v.mean()])
    ax.bar(order, means, yerr=np.array(errs).T, capsize=6, color=["#9aa5b1", "#2b7a78", "#3aafa9"])
    ax.set_title(labels[m]); ax.tick_params(axis="x", rotation=20)
plt.suptitle(f"{MODEL_ID}: closed-book vs RAG (n={N_QUESTIONS}, 95% bootstrap CI)")
plt.tight_layout(); plt.savefig("results.png", dpi=200); plt.show()

## 9. Error analysis — where did RAG help or hurt?
Numbers alone don't make a strong portfolio piece; **examples do**. Look at the biggest wins and losses and note *why* (bad retrieval? model ignored the passages? over-cautious abstention?).

In [ ]:
piv = res.pivot(index="qid", columns="condition", values="semantic_sim")
piv["gain"] = piv["rag"] - piv["closed_book"]
def show(qid):
    sub = res[res.qid == qid].set_index("condition")
    print("Q:", sub.question.iloc[0])
    for c in ["closed_book", "rag"]:
        print(f"\n--- {c} (sim={sub.loc[c,'semantic_sim']:.2f}, support={sub.loc[c,'support_rate']:.2f})")
        print(textwrap.fill(sub.loc[c, "answer"], 110))
    print("\n" + "=" * 110)

print("##### BIGGEST RAG WINS #####")
for q in piv.gain.nlargest(2).index: show(q)
print("##### BIGGEST RAG LOSSES #####")
for q in piv.gain.nsmallest(2).index: show(q)

In [ ]:
# Does RAG help more for some question types?
res.pivot_table(index="qtype", columns="condition", values="semantic_sim", aggfunc="mean")\
   .assign(n=eval_df.qtype.value_counts())\
   .query("n >= 5").sort_values("n", ascending=False).round(3)

## 10. Validate the automatic metrics with human judgement (strongly recommended)
Automatic metrics are proxies. Rating a **blinded** sample yourself — without knowing which condition produced each answer — shows whether the metrics track real quality. This step is what separates a careful study from a demo.

1. Run the next cell; it downloads `human_rating.csv`.
2. Fill the `correct` column: **2** = correct & complete, **1** = partly correct, **0** = wrong/unsafe.
3. Upload the filled file back to Colab and run the cell after.

In [ ]:
sample = res.sample(n=min(30, len(res)), random_state=SEED).reset_index()
blind = sample[["index", "question", "reference", "answer"]].rename(columns={"index": "row_id"})
blind["correct"] = ""
blind.sample(frac=1, random_state=1).to_csv("human_rating.csv", index=False)  # shuffled, condition hidden
try:
    from google.colab import files; files.download("human_rating.csv")
except ImportError:
    pass

In [ ]:
from scipy.stats import spearmanr
rated = pd.read_csv("human_rating.csv").dropna(subset=["correct"])
merged = rated.merge(res.reset_index()[["index", "condition"] + METRICS], left_on="row_id", right_on="index")
print(merged.groupby("condition")["correct"].mean().round(2))
for m in METRICS:
    rho, p = spearmanr(merged["correct"], merged[m], nan_policy="omit")
    print(f"Spearman(human, {m}) = {rho:.2f} (p={p:.3f})")

## 11. Save everything
Download these files for your GitHub repo: `generations.csv`, `results_scored.csv`, `results.png`, and the summary tables.

In [ ]:
summary.to_csv("summary.csv"); tests.to_csv("stat_tests.csv", index=False)
with open("config.json", "w") as f:
    json.dump({k: globals()[k] for k in ["SEED","N_QUESTIONS","MODEL_ID","EMBED_MODEL","EVAL_EMBED",
               "NLI_MODEL","TOP_K","CHUNK_WORDS","CHUNK_OVERLAP","MAX_NEW_TOKENS"]}, f, indent=2)
try:
    from google.colab import files
    for fn in ["generations.csv","results_scored.csv","summary.csv","stat_tests.csv","results.png","config.json"]:
        files.download(fn)
except ImportError:
    pass

## 12. Interpreting results & limitations (put these in your write-up)

**How to answer the research question**
- If `rag` beats `closed_book` on semantic similarity / ROUGE-L with a CI above 0 → support for **H1**.
- If `rag` has a higher support rate and lower contradiction rate → support for **H2**.
- Compare `rag` with `rag_hard`: if the gain shrinks a lot, much of RAG's benefit comes from retrieving the *exact* source document. That is an honest, interesting finding either way.

**Limitations to state openly**
1. **Reference leakage in condition B:** the reference answer is in the library, so condition B is a best case. Condition C exists to address this.
2. **Metric validity:** ROUGE rewards copying; NLI models make errors on medical language. Section 10 checks this.
3. **One small model, one seed, one dataset.** Results may not generalise to larger models.
4. **Source currency:** MedQuAD was collected in the late 2010s; some guidance may be outdated.
5. **Not clinical validation.** No clinician reviewed the answers.

**Extensions (pick one or two for a stronger portfolio)**
- Vary `TOP_K` (1, 3, 5) and plot accuracy vs. number of passages.
- Compare two model sizes (1.5B vs 3B): does RAG help small models more?
- Add a **"distractor" condition** with irrelevant passages to test robustness.
- Use a stronger LLM as a judge (e.g., via an API) and compare with your human ratings.
- Swap in a different trusted source (e.g., WHO fact sheets) for out-of-domain retrieval.